# 04 — Attention and FlashAttention-Style Online Softmax

**Goal:** compare a clear PyTorch attention reference with a Triton implementation that processes keys/values in tiles.

The reference forms the complete attention-score matrix. The Triton version keeps a running maximum, a running normalization sum, and a running output while visiting one key/value tile at a time. This avoids storing the whole score matrix in global memory.

This is a small **educational forward-pass prototype**, not a full replacement for production FlashAttention. It uses non-causal attention and fixed-size head dimensions for clarity.

## 1. Environment

In [ ]:
import math
import statistics
import time
import torch
import triton
import triton.language as tl

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Google Colab, then run this notebook again.")

print("PyTorch:", torch.__version__)
print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## 2. PyTorch reference

For each query, attention calculates `softmax(Q @ K.T / sqrt(head_dim)) @ V`. We will keep the reference in FP32 for easier numerical comparison, then cast the output to the input dtype.

In [ ]:
def torch_attention_reference(q, k, v):
    q32 = q.float()
    k32 = k.float()
    v32 = v.float()
    scores = torch.matmul(q32, k32.transpose(-2, -1)) / math.sqrt(q.shape[-1])
    probabilities = torch.softmax(scores, dim=-1)
    return torch.matmul(probabilities, v32).to(q.dtype)

## 3. Triton kernel with tiled keys/values

Each program handles a block of query rows. It loops over blocks of keys and values. `m_i` stores a running maximum, `l_i` stores a running sum for softmax normalization, and `acc` stores a running weighted sum of values.

When a new tile arrives, the previous running values are rescaled so the result remains mathematically consistent. This is the online-softmax idea used by FlashAttention-like algorithms.

In [ ]:
@triton.jit
def attention_online_kernel(Q_ptr, K_ptr, V_ptr, O_ptr,
                             N_CTX: tl.constexpr, HEAD_DIM: tl.constexpr,
                             BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
                             SCALE: tl.constexpr):
    query_block = tl.program_id(0)
    batch_head = tl.program_id(1)

    rows = query_block * BLOCK_M + tl.arange(0, BLOCK_M)
    key_cols = tl.arange(0, BLOCK_N)
    dims = tl.arange(0, HEAD_DIM)

    q_base = Q_ptr + batch_head * N_CTX * HEAD_DIM
    k_base = K_ptr + batch_head * N_CTX * HEAD_DIM
    v_base = V_ptr + batch_head * N_CTX * HEAD_DIM
    o_base = O_ptr + batch_head * N_CTX * HEAD_DIM

    q_ptrs = q_base + rows[:, None] * HEAD_DIM + dims[None, :]
    q = tl.load(q_ptrs, mask=rows[:, None] < N_CTX, other=0.0)

    running_max = tl.full((BLOCK_M,), -float("inf"), tl.float32)
    running_sum = tl.zeros((BLOCK_M,), tl.float32)
    running_acc = tl.zeros((BLOCK_M, HEAD_DIM), tl.float32)

    for key_start in range(0, N_CTX, BLOCK_N):
        keys = key_start + key_cols
        k_ptrs = k_base + keys[:, None] * HEAD_DIM + dims[None, :]
        v_ptrs = v_base + keys[:, None] * HEAD_DIM + dims[None, :]
        k = tl.load(k_ptrs, mask=keys[:, None] < N_CTX, other=0.0)
        v = tl.load(v_ptrs, mask=keys[:, None] < N_CTX, other=0.0)

        scores = tl.dot(q, tl.trans(k)) * SCALE
        scores = tl.where(keys[None, :] < N_CTX, scores, -float("inf"))

        tile_max = tl.max(scores, axis=1)
        new_max = tl.maximum(running_max, tile_max)
        old_scale = tl.exp(running_max - new_max)
        probabilities = tl.exp(scores - new_max[:, None])
        tile_sum = tl.sum(probabilities, axis=1)

        running_acc = running_acc * old_scale[:, None] + tl.dot(
            probabilities.to(v.dtype), v
        )
        running_sum = running_sum * old_scale + tile_sum
        running_max = new_max

    output = running_acc / running_sum[:, None]
    o_ptrs = o_base + rows[:, None] * HEAD_DIM + dims[None, :]
    tl.store(o_ptrs, output, mask=rows[:, None] < N_CTX)


def triton_attention(q, k, v, out, block_m=32, block_n=64):
    batch, heads, n_ctx, head_dim = q.shape
    if head_dim not in (16, 32, 64, 128):
        raise ValueError("For this demo, head_dim must be 16, 32, 64, or 128")
    if not q.is_contiguous() or not k.is_contiguous() or not v.is_contiguous():
        raise ValueError("The demo expects contiguous Q, K, and V tensors")
    grid = (triton.cdiv(n_ctx, block_m), batch * heads)
    attention_online_kernel[grid](
        q, k, v, out, n_ctx, head_dim, block_m, block_n,
        SCALE=1.0 / math.sqrt(head_dim)
    )
    return out

## 4. Check correctness

We start with a small sequence, two heads, and 64 features per head. Both implementations receive exactly the same tensors.

In [ ]:
BATCH, HEADS, N_CTX, HEAD_DIM = 1, 2, 128, 64
q = torch.randn((BATCH, HEADS, N_CTX, HEAD_DIM), device="cuda", dtype=torch.float16)
k = torch.randn_like(q)
v = torch.randn_like(q)
out = torch.empty_like(q)

triton_attention(q, k, v, out)
expected = torch_attention_reference(q, k, v)
max_abs_error = (out.float() - expected.float()).abs().max().item()
torch.testing.assert_close(out, expected, atol=2e-2, rtol=2e-2)
print("Attention correctness passed")
print("Maximum absolute error:", max_abs_error)

## 5. Compare the two paths

The reference materializes the full score matrix, while the Triton path streams through key/value tiles. The code below is a first comparison, not a claim that this prototype is faster than PyTorch's optimized scaled-dot-product attention. The PyTorch reference also allocates intermediate tensors, so record exactly which path is being timed.

In [ ]:
def median_gpu_time_ms(fn, warmup=10, repeats=50, rounds=5):
    """Measure repeated calls with CUDA events; return median ms per call and all rounds."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    samples = []
    for _ in range(rounds):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(repeats):
            fn()
        end.record()
        end.synchronize()
        samples.append(start.elapsed_time(end) / repeats)

    return statistics.median(samples), samples

In [ ]:
BATCH, HEADS, N_CTX, HEAD_DIM = 1, 2, 128, 64
q = torch.randn((BATCH, HEADS, N_CTX, HEAD_DIM), device="cuda", dtype=torch.float16)
k = torch.randn_like(q)
v = torch.randn_like(q)
out_triton = torch.empty_like(q)

def run_reference():
    return torch_attention_reference(q, k, v)

def run_triton():
    triton_attention(q, k, v, out_triton)

run_reference(); run_triton()
torch.testing.assert_close(out_triton, run_reference(), atol=2e-2, rtol=2e-2)
reference_ms, _ = median_gpu_time_ms(run_reference, repeats=10)
triton_ms, _ = median_gpu_time_ms(run_triton, repeats=10)
print(f"PyTorch reference: {reference_ms:.6f} ms/call")
print(f"Triton prototype:  {triton_ms:.6f} ms/call")
print(f"Reference / Triton ratio: {reference_ms / triton_ms:.3f}x")

## What this does and does not show

This experiment demonstrates the tiled online-softmax idea and checks the resulting output against a reference. It does not include causal masking, backward propagation, grouped-query attention, dropout, or the optimizations found in production FlashAttention. These are possible future extensions—not claims about the current kernel.